Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Load All Three Models + Interactive Input + Three‑Model Comparison

In [3]:
import pandas as pd
import numpy as np
import joblib
import json
import os

SAVE_DIR = '/content/drive/MyDrive/BMDS2003_Project_Models'

# Load models and preprocessing objects
logreg_model = joblib.load(os.path.join(SAVE_DIR, 'logistic_regression_model.pkl'))
dt_model     = joblib.load(os.path.join(SAVE_DIR, 'decision_tree_model.pkl'))
rf_model     = joblib.load(os.path.join(SAVE_DIR, 'random_forest_model.pkl'))
scaler       = joblib.load(os.path.join(SAVE_DIR, 'scaler.pkl'))
target_le    = joblib.load(os.path.join(SAVE_DIR, 'target_label_encoder.pkl'))

with open(os.path.join(SAVE_DIR, 'feature_names.json'), 'r') as f:
    feature_names = json.load(f)

trained_models = {
    "Logistic Regression": logreg_model,
    "Decision Tree": dt_model,
    "Random Forest": rf_model
}

class_names = list(target_le.classes_)

# ============================================================
# Default values for user input (Male, 22y, 1.75m, 70kg)
# ============================================================
DEFAULT_VALUES = {
    'gender': 'Male',
    'age': 22.0,
    'height': 1.75,
    'weight': 70.0,
    'family_history_with_overweight': 'no',
    'favc': 'no',
    'fcvc': 2.0,
    'ncp': 3.0,
    'caec': 'Sometimes',
    'smoke': 'no',
    'ch2o': 3.0,
    'scc': 'no',
    'faf': 1.0,
    'tue': 1.0,
    'calc': 'no',
    'mtrans': 'Public_Transportation'
}

# Default target (index 1 = Normal_Weight)
DEFAULT_TARGET_IDX = 1

# ============================================================
# User input function
# ============================================================
def get_user_input():
    print("\n" + "="*60)
    print("🍽️  OBESITY LEVEL PREDICTOR - MULTI-MODEL COMPARISON 🍽️")
    print("Press Enter to use the [Default] value (based on a real sample).")
    print("="*60)

    user_data = {}

    try:
        gender = input(f"Gender (Female/Male) [{DEFAULT_VALUES['gender']}]: ").strip().capitalize()
        user_data['gender'] = gender if gender in ['Female', 'Male'] else DEFAULT_VALUES['gender']

        age_str = input(f"Age [{DEFAULT_VALUES['age']:.1f}]: ").strip()
        user_data['age'] = float(age_str) if age_str else DEFAULT_VALUES['age']

        height_str = input(f"Height in meters [{DEFAULT_VALUES['height']:.2f}]: ").strip()
        user_data['height'] = float(height_str) if height_str else DEFAULT_VALUES['height']

        weight_str = input(f"Weight in kg [{DEFAULT_VALUES['weight']:.1f}]: ").strip()
        user_data['weight'] = float(weight_str) if weight_str else DEFAULT_VALUES['weight']

        fam = input(f"Family history with overweight (yes/no) [{DEFAULT_VALUES['family_history_with_overweight']}]: ").strip().lower()
        user_data['family_history_with_overweight'] = fam if fam in ['yes', 'no'] else DEFAULT_VALUES['family_history_with_overweight']

        favc = input(f"Frequent high caloric food (yes/no) [{DEFAULT_VALUES['favc']}]: ").strip().lower()
        user_data['favc'] = favc if favc in ['yes', 'no'] else DEFAULT_VALUES['favc']

        fcvc_str = input(f"Vegetable consumption frequency (1-3) [{DEFAULT_VALUES['fcvc']:.1f}]: ").strip()
        user_data['fcvc'] = float(fcvc_str) if fcvc_str else DEFAULT_VALUES['fcvc']

        ncp_str = input(f"Number of main meals per day (1-4) [{DEFAULT_VALUES['ncp']:.1f}]: ").strip()
        user_data['ncp'] = float(ncp_str) if ncp_str else DEFAULT_VALUES['ncp']

        caec = input(f"Food between meals (no/Sometimes/Frequently/Always) [{DEFAULT_VALUES['caec']}]: ").strip().capitalize()
        user_data['caec'] = caec if caec in ['No', 'Sometimes', 'Frequently', 'Always'] else DEFAULT_VALUES['caec']

        ch2o_str = input(f"Daily water intake in liters (1-3) [{DEFAULT_VALUES['ch2o']:.1f}]: ").strip()
        user_data['ch2o'] = float(ch2o_str) if ch2o_str else DEFAULT_VALUES['ch2o']

        smoke = input(f"Do you smoke? (yes/no) [{DEFAULT_VALUES['smoke']}]: ").strip().lower()
        user_data['smoke'] = smoke if smoke in ['yes', 'no'] else DEFAULT_VALUES['smoke']

        scc = input(f"Do you monitor calorie consumption? (yes/no) [{DEFAULT_VALUES['scc']}]: ").strip().lower()
        user_data['scc'] = scc if scc in ['yes', 'no'] else DEFAULT_VALUES['scc']

        faf_str = input(f"Physical activity frequency (0-3 days/week) [{DEFAULT_VALUES['faf']:.1f}]: ").strip()
        user_data['faf'] = float(faf_str) if faf_str else DEFAULT_VALUES['faf']

        tue_str = input(f"Time using technology devices (0-2 hours/day) [{DEFAULT_VALUES['tue']:.1f}]: ").strip()
        user_data['tue'] = float(tue_str) if tue_str else DEFAULT_VALUES['tue']

        calc = input(f"Alcohol consumption (no/Sometimes/Frequently/Always) [{DEFAULT_VALUES['calc']}]: ").strip().capitalize()
        user_data['calc'] = calc if calc in ['No', 'Sometimes', 'Frequently', 'Always'] else DEFAULT_VALUES['calc']

        mtrans = input(f"Transportation (Automobile/Bike/Motorbike/Public_Transportation/Walking) [{DEFAULT_VALUES['mtrans']}]: ").strip()
        user_data['mtrans'] = mtrans if mtrans else DEFAULT_VALUES['mtrans']

    except ValueError:
        print("\n❌ Invalid input. Please enter numbers where expected.")
        return None, None

    # ----- Choose target obesity level (semi-dynamic) -----
    print("\n" + "-"*40)
    print("🎯 TARGET OBESITY LEVEL")
    print("Select the expected obesity level (for hit/miss comparison):")
    for i, name in enumerate(class_names, 1):
        marker = " (default)" if i == DEFAULT_TARGET_IDX + 1 else ""
        print(f"  {i}. {name}{marker}")
    target_choice = input(f"Enter number (1-7) [{DEFAULT_TARGET_IDX+1}]: ").strip()
    if target_choice.isdigit() and 1 <= int(target_choice) <= 7:
        target_class = class_names[int(target_choice)-1]
    else:
        target_class = class_names[DEFAULT_TARGET_IDX]
    print(f"Target Obesity Level: {target_class}")

    return pd.DataFrame([user_data]), target_class

# ============================================================
# Preprocessing function
# ============================================================
def preprocess_input(user_df):
    df = user_df.copy()

    binary_map = {'yes': 1, 'no': 0}
    df['family_history_with_overweight'] = df['family_history_with_overweight'].str.lower().map(binary_map)
    df['favc'] = df['favc'].str.lower().map(binary_map)
    df['smoke'] = df['smoke'].str.lower().map(binary_map)
    df['scc'] = df['scc'].str.lower().map(binary_map)

    df['gender'] = df['gender'].map({'Female': 0, 'Male': 1})

    caec_map = {'No': 0, 'Sometimes': 1, 'Frequently': 2, 'Always': 3}
    calc_map = {'No': 0, 'Sometimes': 1, 'Frequently': 2, 'Always': 3}
    df['caec'] = df['caec'].str.capitalize().map(caec_map)
    df['calc'] = df['calc'].str.capitalize().map(calc_map)

    mtrans_dummies = pd.get_dummies(df['mtrans'], prefix='MTRANS')
    df = pd.concat([df, mtrans_dummies], axis=1)
    df.drop('mtrans', axis=1, inplace=True)

    df_aligned = df.reindex(columns=feature_names, fill_value=0)

    continuous_cols = ['age', 'height', 'weight', 'fcvc', 'ncp', 'ch2o', 'faf', 'tue']
    df_aligned[continuous_cols] = df_aligned[continuous_cols].astype(float)
    df_aligned[continuous_cols] = scaler.transform(df_aligned[continuous_cols])

    if df_aligned.isnull().any().any():
        print("\n⚠️ Warning: NaN values still present!")
        print(df_aligned.isnull().sum()[df_aligned.isnull().sum() > 0])

    return df_aligned

# ============================================================
# Main interactive loop
# ============================================================
def predict_interactive():
    while True:
        user_df, target_class = get_user_input()
        if user_df is None:
            continue

        processed = preprocess_input(user_df)

        print("\n" + "*"*60)
        print("🎯 PREDICTION RESULTS")
        print("*"*60)
        print(f"Target Obesity Level: {target_class}\n")

        for model_name, model in trained_models.items():
            pred_encoded = model.predict(processed)[0]
            pred_label = target_le.inverse_transform([pred_encoded])[0]
            proba = model.predict_proba(processed)[0]
            max_prob = np.max(proba)

            if pred_label == target_class:
                status = "✅ TARGET HIT"
            else:
                status = "⚠️ TARGET MISSED"

            print(f"{model_name:>20}: {pred_label:25s} (Confidence: {max_prob:.2%})  |  {status}")

        print("*"*60)

        again = input("\nWould you like to test another scenario? (y/n): ").strip().lower()
        if again != 'y':
            print("Exiting interactive mode. Stay healthy!")
            break

# Start the predictor
predict_interactive()



🍽️  OBESITY LEVEL PREDICTOR - MULTI-MODEL COMPARISON 🍽️
Press Enter to use the [Default] value (based on a real sample).
Gender (Female/Male) [Male]: 
Age [22.0]: 
Height in meters [1.75]: 
Weight in kg [70.0]: 
Family history with overweight (yes/no) [no]: 
Frequent high caloric food (yes/no) [no]: 
Vegetable consumption frequency (1-3) [2.0]: 
Number of main meals per day (1-4) [3.0]: 
Food between meals (no/Sometimes/Frequently/Always) [Sometimes]: 
Daily water intake in liters (1-3) [3.0]: 
Do you smoke? (yes/no) [no]: 
Do you monitor calorie consumption? (yes/no) [no]: 
Physical activity frequency (0-3 days/week) [1.0]: 
Time using technology devices (0-2 hours/day) [1.0]: 
Alcohol consumption (no/Sometimes/Frequently/Always) [no]: 
Transportation (Automobile/Bike/Motorbike/Public_Transportation/Walking) [Public_Transportation]: 

----------------------------------------
🎯 TARGET OBESITY LEVEL
Select the expected obesity level (for hit/miss comparison):
  1. Insufficient_Weight
 

OPTIMAL CONDITIONS BY MODEL

In [4]:
import pandas as pd
import numpy as np
import joblib
import json
import os
from google.colab import drive

# ============================================================
# 1. Mount Drive and load models & preprocessing objects
# ============================================================
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/BMDS2003_Project_Models'

logreg_model = joblib.load(os.path.join(SAVE_DIR, 'logistic_regression_model.pkl'))
dt_model     = joblib.load(os.path.join(SAVE_DIR, 'decision_tree_model.pkl'))
rf_model     = joblib.load(os.path.join(SAVE_DIR, 'random_forest_model.pkl'))
scaler       = joblib.load(os.path.join(SAVE_DIR, 'scaler.pkl'))
target_le    = joblib.load(os.path.join(SAVE_DIR, 'target_label_encoder.pkl'))

with open(os.path.join(SAVE_DIR, 'feature_names.json'), 'r') as f:
    feature_names = json.load(f)

trained_models = {
    "Logistic Regression": logreg_model,
    "Decision Tree": dt_model,
    "Random Forest": rf_model
}

class_names = list(target_le.classes_)

# ============================================================
# 2. Let user choose the target class dynamically
# ============================================================
print("\n" + "="*60)
print(" SELECT TARGET OBESITY LEVEL ")
print("="*60)
for i, name in enumerate(class_names, 1):
    print(f"  {i}. {name}")
while True:
    choice = input("Enter number (1-7): ").strip()
    if choice.isdigit() and 1 <= int(choice) <= 7:
        TARGET_CLASS = class_names[int(choice)-1]
        break
    else:
        print("❌ Invalid. Please enter a number between 1 and 7.")

# ============================================================
# 3. Configuration for simulation
# ============================================================
N_SIMULATIONS = 10000
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ============================================================
# 4. Generate random realistic inputs
# ============================================================
print(f"\nSimulating {N_SIMULATIONS} random lifestyle profiles for target: {TARGET_CLASS}...")

sim_data = {
    'gender': np.random.choice(['Female', 'Male'], N_SIMULATIONS),
    'age': np.random.uniform(14, 65, N_SIMULATIONS),
    'height': np.random.uniform(1.45, 2.00, N_SIMULATIONS),
    'weight': np.random.uniform(40, 180, N_SIMULATIONS),
    'family_history_with_overweight': np.random.choice(['yes', 'no'], N_SIMULATIONS),
    'favc': np.random.choice(['yes', 'no'], N_SIMULATIONS),
    'fcvc': np.random.uniform(1.0, 3.0, N_SIMULATIONS),
    'ncp': np.random.uniform(1.0, 4.0, N_SIMULATIONS),
    'caec': np.random.choice(['no', 'Sometimes', 'Frequently', 'Always'], N_SIMULATIONS),
    'ch2o': np.random.uniform(1.0, 3.0, N_SIMULATIONS),
    'smoke': np.random.choice(['yes', 'no'], N_SIMULATIONS),
    'scc': np.random.choice(['yes', 'no'], N_SIMULATIONS),
    'faf': np.random.uniform(0.0, 3.0, N_SIMULATIONS),
    'tue': np.random.uniform(0.0, 2.0, N_SIMULATIONS),
    'calc': np.random.choice(['no', 'Sometimes', 'Frequently', 'Always'], N_SIMULATIONS),
    'mtrans': np.random.choice(['Automobile', 'Bike', 'Motorbike',
                                'Public_Transportation', 'Walking'], N_SIMULATIONS)
}
sim_df = pd.DataFrame(sim_data)

# ============================================================
# 5. Preprocess the simulated data (exactly like training)
# ============================================================
def preprocess_simulation(df):
    df = df.copy()
    binary_map = {'yes': 1, 'no': 0}
    df['gender'] = df['gender'].map({'Female': 0, 'Male': 1})
    df['family_history_with_overweight'] = df['family_history_with_overweight'].map(binary_map)
    df['favc'] = df['favc'].map(binary_map)
    df['smoke'] = df['smoke'].map(binary_map)
    df['scc'] = df['scc'].map(binary_map)

    caec_map = {'no': 0, 'Sometimes': 1, 'Frequently': 2, 'Always': 3}
    calc_map = {'no': 0, 'Sometimes': 1, 'Frequently': 2, 'Always': 3}
    df['caec'] = df['caec'].map(caec_map)
    df['calc'] = df['calc'].map(calc_map)

    mtrans_dummies = pd.get_dummies(df['mtrans'], prefix='MTRANS')
    df = pd.concat([df, mtrans_dummies], axis=1)
    df.drop('mtrans', axis=1, inplace=True)

    df_aligned = df.reindex(columns=feature_names, fill_value=0)
    continuous_cols = ['age', 'height', 'weight', 'fcvc', 'ncp', 'ch2o', 'faf', 'tue']
    df_aligned[continuous_cols] = scaler.transform(df_aligned[continuous_cols])
    return df_aligned

sim_processed = preprocess_simulation(sim_df)

# ============================================================
# 6. Find optimal conditions for each model
# ============================================================
target_idx = class_names.index(TARGET_CLASS)

print("\n" + "="*60)
print(" OPTIMAL CONDITIONS BY MODEL ".center(60))
print(f"Target Class: {TARGET_CLASS}")
print("="*60)

for model_name, model in trained_models.items():
    proba = model.predict_proba(sim_processed)[:, target_idx]
    best_idx = np.argmax(proba)
    best_prob = proba[best_idx]
    best_row = sim_df.iloc[best_idx]

    print(f"\n>>> {model_name}")
    print(f"    Maximum Confidence for '{TARGET_CLASS}': {best_prob:.2%}")
    print("    " + "-"*40)
    print(f"    Gender                : {best_row['gender']}")
    print(f"    Age                   : {best_row['age']:.2f}")
    print(f"    Height (m)            : {best_row['height']:.2f}")
    print(f"    Weight (kg)           : {best_row['weight']:.1f}")
    print(f"    Family History        : {best_row['family_history_with_overweight']}")
    print(f"    High Caloric Food     : {best_row['favc']}")
    print(f"    Vegetables (FCVC)     : {best_row['fcvc']:.1f}")
    print(f"    Main Meals (NCP)      : {best_row['ncp']:.1f}")
    print(f"    Food Between Meals    : {best_row['caec']}")
    print(f"    Water Intake (CH2O)   : {best_row['ch2o']:.2f}")
    print(f"    Smoker                : {best_row['smoke']}")
    print(f"    Calorie Monitoring    : {best_row['scc']}")
    print(f"    Physical Activity     : {best_row['faf']:.2f}")
    print(f"    Tech Use (TUE)        : {best_row['tue']:.2f}")
    print(f"    Alcohol (CALC)        : {best_row['calc']}")
    print(f"    Transportation        : {best_row['mtrans']}")

print("\n" + "="*60)
print("Simulation complete.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

 SELECT TARGET OBESITY LEVEL 
  1. Insufficient_Weight
  2. Normal_Weight
  3. Obesity_Type_I
  4. Obesity_Type_II
  5. Obesity_Type_III
  6. Overweight_Level_I
  7. Overweight_Level_II
Enter number (1-7): 2

Simulating 10000 random lifestyle profiles for target: Normal_Weight...

                OPTIMAL CONDITIONS BY MODEL                 
Target Class: Normal_Weight

>>> Logistic Regression
    Maximum Confidence for 'Normal_Weight': 99.49%
    ----------------------------------------
    Gender                : Male
    Age                   : 47.63
    Height (m)            : 1.52
    Weight (kg)           : 52.3
    Family History        : no
    High Caloric Food     : no
    Vegetables (FCVC)     : 2.7
    Main Meals (NCP)      : 3.7
    Food Between Meals    : Always
    Water Intake (CH2O)   : 2.34
    Smoker                : yes
    Calorie Monitor